# Test Model Serving via MaaS

This notebook tests MaaS model serving capabilities:
1. Model discovery via MaaS API
2. Inference with API key authentication
3. Streaming support
4. Rate limiting behavior
5. Concurrent requests

In [ ]:
import subprocess
import json
import time
import os

result = subprocess.run(
    ["kubectl", "get", "ingresses.config.openshift.io", "cluster",
     "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
)
CLUSTER_DOMAIN = result.stdout.strip()
MAAS_HOST = f"https://maas.{CLUSTER_DOMAIN}"

# Use API key if available, otherwise fall back to OCP token
API_KEY = os.getenv("MAAS_API_KEY", "")
if not API_KEY:
    token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
    API_KEY = token_result.stdout.strip()
    print("Using OpenShift token for authentication")
else:
    print(f"Using MaaS API key: {API_KEY[:15]}...")

print(f"\n✅ MaaS Gateway: {MAAS_HOST}")

## 1. Model Discovery

List all models available through MaaS.

In [ ]:
import urllib.request
import ssl

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(
    f"{MAAS_HOST}/maas-api/v1/models",
    headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
)
with urllib.request.urlopen(req, context=ctx) as resp:
    models_data = json.loads(resp.read())

print("Available Models via MaaS")
print("=" * 60)
print(f"{'Model ID':<30} {'URL'}")
print("-" * 60)

for model in models_data.get("data", []):
    print(f"{model['id']:<30} {model.get('url', 'N/A')}")

# Save first model for subsequent tests
if models_data.get("data"):
    TEST_MODEL = models_data["data"][0]["id"]
    TEST_URL = models_data["data"][0]["url"]
    print(f"\nUsing '{TEST_MODEL}' for tests")
else:
    print("\n⚠️  No models found. Deploy a model with MaaS enabled first.")

## 2. Inference Test

Send a chat completion request through MaaS.

In [ ]:
# pip install openai
from openai import OpenAI

client = OpenAI(
    base_url=f"{TEST_URL}/v1",
    api_key=API_KEY,
    http_client=None
)

print(f"Testing inference on '{TEST_MODEL}'...")
print("-" * 40)

start = time.time()
response = client.chat.completions.create(
    model=TEST_MODEL,
    messages=[{"role": "user", "content": "Write a Python hello world in one line."}],
    max_tokens=50
)
elapsed = time.time() - start

print(f"Model: {response.model}")
print(f"Response: {response.choices[0].message.content.strip()}")
print(f"Tokens: {response.usage.prompt_tokens} in / {response.usage.completion_tokens} out")
print(f"Latency: {elapsed:.1f}s")

## 3. Streaming Support

Verify streaming works through MaaS (critical for IDE integration).

In [ ]:
print(f"Streaming response from '{TEST_MODEL}':")
print("-" * 40)

start = time.time()
stream = client.chat.completions.create(
    model=TEST_MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 5, one per line."}],
    max_tokens=50,
    stream=True
)

first_token_time = None
for chunk in stream:
    if chunk.choices[0].delta.content:
        if first_token_time is None:
            first_token_time = time.time() - start
        print(chunk.choices[0].delta.content, end="", flush=True)

total_time = time.time() - start
print(f"\n\n✅ Streaming works")
print(f"   Time to first token: {first_token_time:.2f}s")
print(f"   Total time: {total_time:.2f}s")

## 4. Rate Limiting

Send rapid requests to observe MaaS rate limiting behavior.
Expect 200 OK initially, followed by 429 when limits are exceeded.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas.${CLUSTER_DOMAIN}"

MODELS_JSON=$(curl -sSk "${HOST}/maas-api/v1/models" \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json")
MODEL_NAME=$(echo $MODELS_JSON | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'][0]['id'] if d.get('data') else '')")
MODEL_URL=$(echo $MODELS_JSON | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'][0]['url'] if d.get('data') else '')")

echo "Sending 10 rapid requests to test rate limiting..."
echo ""

for i in $(seq 1 10); do
  HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" \
    -H "Authorization: Bearer $(oc whoami -t)" \
    -H "Content-Type: application/json" \
    -d "{\"model\": \"${MODEL_NAME}\", \"messages\": [{\"role\": \"user\", \"content\": \"Hi\"}], \"max_tokens\": 5}" \
    "${MODEL_URL}/v1/chat/completions")
  echo "  Request $i: HTTP $HTTP_CODE"
done

## 5. Concurrent Requests

Simulate multiple developers using MaaS simultaneously.

In [ ]:
import concurrent.futures

def make_request(i):
    start = time.time()
    try:
        response = client.chat.completions.create(
            model=TEST_MODEL,
            messages=[{"role": "user", "content": f"Say {i}"}],
            max_tokens=5
        )
        elapsed = time.time() - start
        return f"✅ Request {i}: {response.model} ({elapsed:.1f}s)"
    except Exception as e:
        elapsed = time.time() - start
        return f"❌ Request {i}: {str(e)[:40]} ({elapsed:.1f}s)"

print("Sending 5 concurrent requests:")
print("-" * 50)

with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(make_request, i) for i in range(5)]
    for future in concurrent.futures.as_completed(futures):
        print(f"  {future.result()}")

## Summary

| Test | What It Validates |
|------|-------------------|
| Model Discovery | MaaS API returns available models |
| Inference | Chat completion works through MaaS gateway |
| Streaming | Token-by-token streaming for IDE integration |
| Rate Limiting | Subscription-based rate limits enforced |
| Concurrent | Multiple users can access models simultaneously |

## Next Steps

→ `4_test_mcp_servers.ipynb` — Test MCP server access through the MaaS gateway
→ `5_ide_configuration.ipynb` — Configure your IDE to use MaaS endpoints
→ `6_maas_advanced.ipynb` — Explore subscriptions, rate limit tuning, API key management, and monitoring